In [1]:
import os
import math
from datasets import load_dataset
from PIL import Image
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoProcessor, LlavaForConditionalGeneration, AutoTokenizer, LlavaModel

class CustomLlavaModel(LlavaModel):
    def __init__(self, config):
        super().__init__(config)
        # nothing else to add here
        self.counter = 0
        self.last_n_heads = 9
        self.attention_threshold = 0.0004

    def get_image_features(
        self,
        pixel_values: torch.FloatTensor,
        vision_feature_layer=None,
        vision_feature_select_strategy=None,
        **kwargs,
    ):
        kwargs = {k: v for k, v in kwargs.items() if v is not None}

        vision_feature_layer = (
            vision_feature_layer if vision_feature_layer is not None else self.config.vision_feature_layer
        )
        
        image_outputs = self.vision_tower(pixel_values, 
                                          output_hidden_states=True, 
                                          output_attentions=True,
                                          **kwargs)

        all_attns = torch.stack(image_outputs.attentions, dim=0)

        image_outputs = image_outputs.hidden_states[vision_feature_layer]
        image_outputs = image_outputs[:, 1:]

        # image_outputs = self.average_and_mask_attn_heads(image_outputs, all_attns)
        # image_outputs = self.mask_heads_individually(image_outputs, all_attns)
        image_outputs = self.mask_heads_structured(image_outputs, all_attns)
        
        return self.multi_modal_projector(image_outputs) 

    def average_and_mask_attn_heads(self, image_outputs, all_attns):
        cls_to_patch = all_attns[..., 0, 1:]
        cls_to_patch_last = cls_to_patch[:, :, -self.last_n_heads:, :]

        avg_scores = cls_to_patch_last.mean(dim=(0, 2))

        mask = (avg_scores.unsqueeze(-1) >= self.attention_threshold)
        
        image_outputs = image_outputs * mask     

        return image_outputs

    def mask_heads_individually(self, image_feats: torch.Tensor, all_attns: torch.Tensor) -> torch.Tensor:
        """
        Case 1: Mask each of the last N heads individually based on its own attention scores.
        """
        # Extract cls-to-patch: [L, B, H, P]
        cls_to_patch = all_attns[..., 0, 1:]
        # Select last N heads: [L, B, H', P]
        cls_last = cls_to_patch[:, :, -self.last_n_heads:, :]
        # Average over layers: [B, H', P]
        layer_avg = cls_last.mean(dim=0)
        # Build per-head mask: [B, H', P]
        mask_hp = (layer_avg >= self.attention_threshold)

        B, P, hidden = image_feats.shape
        num_heads_total = cls_to_patch.shape[2]
        head_dim = hidden // num_heads_total

        # Create a full-head mask placeholder: [B, P, H]
        head_mask = torch.zeros((B, P, num_heads_total), device=image_feats.device, dtype=mask_hp.dtype)
        # Align mask_hp ([B, H', P]) to head_mask's slice ([B, P, H']) by permuting
        head_mask[:, :, -self.last_n_heads:] = mask_hp.permute(0, 2, 1)

        # Expand head mask to feature dimension: [B, P, H, head_dim] -> [B, P, hidden]
        mask_feat = head_mask.unsqueeze(-1).repeat(1, 1, 1, head_dim).view(B, P, hidden)

        return image_feats * mask_feat

    def mask_heads_structured(self, image_feats: torch.Tensor, all_attns: torch.Tensor) -> torch.Tensor:
        """
        Case 2: Structured pruning — prune entire attention head features
        if the average cls-to-patch attention of the head is below threshold.
        Logs which heads were pruned.
        """
        # Extract cls-to-patch: [L, B, H, P]
        cls_to_patch = all_attns[..., 0, 1:]
    
        # Select last N heads: [L, B, H', P]
        cls_last = cls_to_patch[:, :, -self.last_n_heads:, :]
    
        # Average attention per head: [B, H', P] -> [H'] after mean over batch and patches
        avg_attn_per_head = cls_last.mean(dim=(0, 3))  # [H']
    
        # Compute binary mask: 1 if head should be kept, 0 if pruned
        head_keep_mask = (avg_attn_per_head >= self.attention_threshold).float()  # [H']
    
        # Determine which heads (relative to full model) were pruned
        num_heads_total = cls_to_patch.shape[2]
        last_head_indices = list(range(num_heads_total - self.last_n_heads, num_heads_total))
    
        pruned_heads = [
            idx for keep, idx in zip(head_keep_mask, last_head_indices) if keep.item() == 0.0
        ]
        print(f"Structured pruning: heads pruned due to low attention: {pruned_heads}")
    
        # Build full mask for all heads: [H]
        head_mask_full = torch.ones(num_heads_total, device=image_feats.device)
        head_mask_full[-self.last_n_heads:] = head_keep_mask
    
        # Expand to feature dimension
        B, P, hidden = image_feats.shape
        head_dim = hidden // num_heads_total
    
        # Create structured mask: [H, head_dim] -> [hidden]
        head_mask_expanded = head_mask_full.unsqueeze(1).repeat(1, head_dim).reshape(-1)  # [hidden]
    
        # Apply structured mask to image_feats: [B, P, hidden]
        return image_feats * head_mask_expanded.view(1, 1, hidden)


2025-07-21 16:37:04.038230: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753115824.266548      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753115824.332689      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [2]:
class CustomLlavaForConditionalGeneration(LlavaForConditionalGeneration):
    def __init__(self, config):
        super().__init__(config)
        # replace the internal model with your custom version
        self.model = CustomLlavaModel(config)

    def forward(
        self,
        input_ids=None,
        pixel_values=None,
        attention_mask=None,
        position_ids=None,
        past_key_values=None,
        inputs_embeds=None,
        vision_feature_layer=None,
        vision_feature_select_strategy=None,
        labels=None,
        use_cache=None,
        output_attentions=None,
        output_hidden_states=None,
        return_dict=None,
        cache_position=None,
        logits_to_keep=0,
        image_sizes=None,
        **kwargs
    ):
        # everything else is exactly the same as the original
        return super().forward(
            input_ids=input_ids,
            pixel_values=pixel_values,
            attention_mask=attention_mask,
            position_ids=position_ids,
            past_key_values=past_key_values,
            inputs_embeds=inputs_embeds,
            vision_feature_layer=vision_feature_layer,
            vision_feature_select_strategy=vision_feature_select_strategy,
            labels=labels,
            use_cache=use_cache,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
            cache_position=cache_position,
            logits_to_keep=logits_to_keep,
            image_sizes=image_sizes,
            **kwargs
        )

In [3]:
model_id = "llava-hf/llava-1.5-7b-hf"
NUM_GPUS = torch.cuda.device_count()
DEVICE = "cuda:0" if NUM_GPUS > 0 else "cpu"
OUTPUT_DIR = "./llava_single_coco_layered_attention"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Using device: {DEVICE} (GPUs available: {NUM_GPUS})")

# %%
# 4. Load Processor and Model
model = CustomLlavaForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if NUM_GPUS > 0 else torch.float32,
    low_cpu_mem_usage=True,
    trust_remote_code=True
)

# model = LlavaForConditionalGeneration.from_pretrained(
#     model_id,
#     torch_dtype=torch.float16 if NUM_GPUS > 0 else torch.float32,
#     low_cpu_mem_usage=True,
#     trust_remote_code=True
# )

tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=False)

processor = AutoProcessor.from_pretrained(model_id)
model = model.half().to(DEVICE)
if NUM_GPUS > 1:
    model = nn.DataParallel(model)
print("Model loaded and ready.")

# %%
# Helper: Unwrap DataParallel

def _get_base_model(m):
    return m.module if isinstance(m, nn.DataParallel) else m

# %%
# 5. Extraction: embeddings + full-layer attentions

def extract_embeddings_and_all_layer_attentions(image, prompt):
    base = _get_base_model(model)
    conv = [{"role": "user", "content": [{"type": "text", "text": prompt}, {"type": "image"}]}]
    text_prompt = processor.apply_chat_template(conv, add_generation_prompt=True)
    inputs = processor(images=image, text=text_prompt, return_tensors="pt", padding=True).to(DEVICE)
    vision = base.vision_tower if hasattr(base, "vision_tower") else base.vision_encoder
    outputs = vision(
        pixel_values=inputs.pixel_values,
        output_hidden_states=True,
        output_attentions=True
    )
    patch_emb = outputs.hidden_states[-1][:, 1:, :].detach().cpu().numpy()[0]
    all_attns = [att.detach().cpu() for att in outputs.attentions]
    return patch_emb, all_attns

# %%
# 6. Compute similarity metrics

def compute_similarity_metrics(embeddings):
    normed = embeddings / (np.linalg.norm(embeddings, axis=1, keepdims=True) + 1e-12)
    cos_sim = normed @ normed.T
    p = np.abs(embeddings)
    p /= (p.sum(axis=1, keepdims=True) + 1e-12)
    size = p.shape[0]
    kl = np.zeros((size, size))
    for i in range(size):
        for j in range(size):
            kl[i, j] = np.sum(p[i] * (np.log(p[i] + 1e-12) - np.log(p[j] + 1e-12)))
    return cos_sim, kl

# %%
# 7. Aggregate attention per layer for CLS-to-patch tokens

def cls_to_patch_attention_per_layer(attentions):
    scores = []
    for layer_attn in attentions:
        attn_mean = layer_attn.mean(dim=1)[0]
        cls_attn = attn_mean[0, 1:].numpy()
        scores.append(cls_attn)
    return np.stack(scores, axis=0)

# %%
# 8. Dynamic visualization: plot attention overlay for a given layer

def plot_attention_map_for_layer(image, layer_idx, scores):
    num_patches = scores.shape[0]
    grid_size = int(math.sqrt(num_patches))
    assert grid_size * grid_size == num_patches, f"Cannot form square grid for {num_patches} patches"
    patch_size = image.size[0] // grid_size
    img_arr = np.array(image.resize((grid_size * patch_size, grid_size * patch_size)))
    heatmap = scores.reshape((grid_size, grid_size))
    plt.figure(figsize=(5,5))
    plt.imshow(img_arr)
    plt.imshow(heatmap, alpha=0.6, cmap='jet', extent=(0, img_arr.shape[1], img_arr.shape[0], 0))
    plt.axis('off')
    plt.title(f"Layer {layer_idx} CLS-to-Patch Attention")
    plt.show()

def compute_head_entropies(attentions):
    # returns array shape (num_layers, num_heads)
    ent = []
    for layer_attn in attentions:
        # layer_attn: [1, heads, seq_len, seq_len]
        h = layer_attn.shape[1]
        head_ent = []
        for head in range(h):
            # CLS->patch distribution
            scores = layer_attn[0, head, 0, 1:].numpy()
            # normalize to probability distribution
            total = scores.sum()
            if total <= 0:
                # all zeros: entropy zero
                head_ent.append(0.0)
                continue
            p = scores / total
            # compute entropy over non-zero entries to avoid log(0)
            nonzero = p > 0
            ent_val = -np.sum(p[nonzero] * np.log(p[nonzero]))
            head_ent.append(ent_val)
        ent.append(head_ent)
    return np.array(ent)


def linear_CKA(X, Y):
    # X, Y: n_samples x n_features
    Xc = X - X.mean(0, keepdims=True)
    Yc = Y - Y.mean(0, keepdims=True)
    K = Xc @ Xc.T
    L = Yc @ Yc.T
    # center
    n = K.shape[0]
    H = np.eye(n) - np.ones((n,n))/n
    Kc = H @ K @ H
    Lc = H @ L @ H
    hsic = np.trace(Kc @ Lc)
    var_k = np.trace(Kc @ Kc)
    var_l = np.trace(Lc @ Lc)
    return hsic / np.sqrt(var_k * var_l)


def compute_cka_per_layer(attentions):
    # returns list of (heads x heads) CKA matrices
    cka_mats = []
    for layer_attn in attentions:
        h = layer_attn.shape[1]
        # use CLS->patch vectors of shape (patches,)
        vecs = []
        for head in range(h):
            scores = layer_attn[0, head, 0, 1:].numpy()
            vecs.append(scores[:, None])
        vecs = np.concatenate(vecs, axis=1)  # patches x heads
        mats = np.zeros((h,h))
        for i in range(h):
            for j in range(h):
                mats[i, j] = linear_CKA(vecs[:, i:i+1], vecs[:, j:j+1])
        cka_mats.append(mats)
    return cka_mats

prompts = [
    "I want to see the lochness monster from this image"
]

# coco = load_dataset("lmms-lab/COCO-Caption", split="val")
# item = coco[81]
# image = item['image'] if 'image' in item else Image.open(item['file_path'])

import requests
from PIL import Image

image_file = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(image_file, stream=True).raw)

gen_model = model.module if isinstance(model, nn.DataParallel) else model
gen_model.config.output_attentions = True

def plot_image_attention(img, prompt, ax):
    # prepend the <img> token so processor inserts the image placeholder
    text_with_img = "<img> " + prompt

    # tokenize + encode image
    inputs = processor(
        images=img,
        text=text_with_img,
        return_tensors="pt",
        padding=True
    ).to(DEVICE)

    output = gen_model.generate(
        input_ids=inputs["input_ids"],
        pixel_values=inputs["pixel_values"],
        attention_mask=inputs.get("attention_mask", None),
        max_new_tokens=20,
        return_dict_in_generate=True,
        output_attentions=True
    )

    all_ca = torch.stack(output.cross_attentions)            # [layers, B, heads, tgt, src]
    avg_ca = all_ca.mean(dim=(0, 2, 3))[0]                   # [src_len]

    # compute how many image tokens
    total_src = avg_ca.size(0)
    n_text = inputs["input_ids"].size(1)
    n_image = total_src - n_text

    # reshape the last n_image weights into square grid
    weights = avg_ca[-n_image:].cpu().reshape(int(n_image**0.5), -1).detach()

    # plot the image + heatmap overlay
    ax.imshow(img)
    ax.imshow(weights.numpy(), cmap="jet", alpha=0.5,
              extent=(0, img.size[0], img.size[1], 0))
    ax.set_title(prompt)
    ax.axis("off")

def _get_base_model(m):
    return m.module if isinstance(m, nn.DataParallel) else m

def _get_cross_attn_modules(model):
    cross_attn_modules = []
    for name, module in model.named_modules():
        if "model.language_model.layers" in name and name.endswith("self_attn"):
            cross_attn_modules.append((name, module))
            
    return cross_attn_modules

def generate_with_hooks(image, prompt, max_new_tokens=50):
    base = _get_base_model(model)

    conv = [{"role": "user", "content": [{"type": "text", "text": prompt}, {"type": "image"}]}]
    text_prompt = processor.apply_chat_template(conv, add_generation_prompt=True)
    inputs = processor(images=image, text=text_prompt, return_tensors="pt", padding=True).to(DEVICE)

    attn_store = {"weights": []}
    hooks = []
    for name, cross_attn in _get_cross_attn_modules(model):
        def _hook(module, inp, output):
            if isinstance(output, tuple) and len(output) >= 2:
                weights = output[1]
            else:
                weights = getattr(module, "attn_weights", None)
            if weights is not None:
                attn_store["weights"].append(weights.detach().cpu())
        hooks.append(cross_attn.register_forward_hook(_hook))

    gen_output = base.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        output_attentions=False,      # we don’t need HF to store them
        return_dict_in_generate=True,
        use_cache=False,              # IMPORTANT so all layers recompute & hook fires
    )

    for h in hooks:
        h.remove()

    text = tokenizer.decode(gen_output.sequences[0], skip_special_tokens=True)
    
    return text, attn_store["weights"], inputs

def plot_attention(image: Image.Image, attn_map: torch.Tensor, patch_size=16, suptitle=None):
    H, W = image.size[1], image.size[0]
    num_patches = attn_map.shape[-1]
    grid_size = int(np.sqrt(num_patches))
    grid = attn_map.reshape(grid_size, grid_size).cpu().numpy()
    heatmap = np.kron(grid, np.ones((patch_size, patch_size)))
    heatmap = heatmap[:H, :W]
    plt.figure(figsize=(6,6))
    plt.imshow(image)
    plt.imshow(heatmap, cmap='jet', alpha=0.5)
    if suptitle:
        plt.title(suptitle)
    plt.axis('off')
    plt.show()

patch_emb, all_attns = extract_embeddings_and_all_layer_attentions(image, prompts[0])
layer_patch_scores = cls_to_patch_attention_per_layer(all_attns)

num_layers = layer_patch_scores.shape[0]

print("vision tower layers", num_layers)
# for l in range(0, num_layers, 2):
#     plot_attention_map_for_layer(image, l, layer_patch_scores[l])

from math import sqrt, floor

num_layers = len(_get_cross_attn_modules(model))

for prompt in prompts:
    resp, all_weights, inputs = generate_with_hooks(image, prompt, max_new_tokens=30)
    print(f"\nPrompt: {prompt!r}\nResponse: {resp}\n")

    # all_weights: flat list of weight tensors, one per layer per generation step
    # Extract only the last generation step's weights:
    last_step_weights = all_weights[-num_layers:]  # list of [batch, n_heads, tgt_len, src_len]

    for layer_idx, layer_w in enumerate(last_step_weights):
        # pick final token's attention
        w_final = layer_w[0, :, -1, :]  # [n_heads, src_len]
        avg_w = w_final.mean(0)         # [src_len]

        # Separate image patch attentions from text tokens
        num_patches = avg_w.shape[0]
        grid_size = int(sqrt(num_patches))
        num_img_patches = grid_size * grid_size
        img_attn = avg_w[:num_img_patches]  # (grid_size^2,)

        # reshape into grid
        attn_grid = img_attn.cpu().numpy().reshape(grid_size, grid_size)

        # compute patch size in pixels
        W, H = image.size
        patch_w = floor(W / grid_size)
        patch_h = floor(H / grid_size)

        # upsample to full resolution
        heatmap = np.kron(attn_grid, np.ones((patch_h, patch_w)))
        heatmap = heatmap[:H, :W]

        # plot per layer
        plt.figure(figsize=(6, 6))
        plt.imshow(image)
        plt.imshow(heatmap, cmap='jet', alpha=0.5)
        plt.axis('off')
        plt.title(f"Cross-Attn Overlay – Prompt: {prompt!r} – Layer {layer_idx}")
        plt.show()

# Summary & Saving
# print(f"Image idx: 10 | #patches: {patch_emb.shape[0]}")
# print(f"Avg Cosine Sim: {cos_sim.mean():.4f} | Avg KL Div: {kl_div.mean():.4f}")
# print(f"Attention shape: {layer_patch_scores.shape}")
# np.save(os.path.join(OUTPUT_DIR, 'coco_cos_sim.npy'), cos_sim)
# np.save(os.path.join(OUTPUT_DIR, 'coco_kl_div.npy'), kl_div)
# np.save(os.path.join(OUTPUT_DIR, 'layer_patch_scores.npy'), layer_patch_scores)

# # %%
# # 10. Visualize inter-token similarity matrices
# plt.figure(figsize=(6,5))
# sns.heatmap(cos_sim, cmap='viridis')
# plt.title('Patch Embedding Cosine Similarity')
# plt.xlabel('Patch Index')
# plt.ylabel('Patch Index')
# plt.show()

# head_entropies = compute_head_entropies(all_attns)
# cka_matrices = compute_cka_per_layer(all_attns)
# # Plot entropy heatmap for layer 0
# plt.figure(figsize=(6,4))
# sns.heatmap(head_entropies, cmap='Reds', xticklabels=[f'H{h}' for h in range(head_entropies.shape[1])], yticklabels=[f'L{l}' for l in range(head_entropies.shape[0])])
# plt.title('Head Entropy across Layers')
# plt.xlabel('Head Index')
# plt.ylabel('Layer Index')
# plt.show()

# # Plot CKA for a chosen layer (e.g., middle)

# for i in range(len(cka_matrics)):
#     plt.figure(figsize=(6,5))
#     sns.heatmap(cka_matrices[i], cmap='coolwarm')
#     plt.title(f'CKA between Heads (Layer {i})')
#     plt.xlabel('Head Index')
#     plt.ylabel('Head Index')
#     plt.show()

# # %%
# # 12. Compute and plot attention entropy per layer
# # Entropy of CLS-to-patch distribution: H = -sum p * log p
# entropies = -np.sum(layer_patch_scores * np.log(layer_patch_scores + 1e-12), axis=1)
# plt.figure(figsize=(6,4))
# plt.plot(range(len(entropies)), entropies, marker='o')
# plt.title('Attention Entropy per Layer')
# plt.xlabel('Layer Index')
# plt.ylabel('Entropy (nats)')
# plt.grid(True)
# plt.show()

# # %%
# # 13. (Optional) Inspect full token attention map for specific layer/head
# # e.g., layer 0, head 0
# layer_idx, head_idx = 0, 0
# full_attn = all_attns[layer_idx][0, head_idx, 1:, 1:].numpy()  # patch-to-patch
# plt.figure(figsize=(5,5))
# sns.heatmap(full_attn, cmap='Blues')
# plt.title(f'Layer {layer_idx} Head {head_idx} Patch-to-Patch Attention')
# plt.xlabel('Patch Index')
# plt.ylabel('Patch Index')
# plt.show()

Using device: cuda:0 (GPUs available: 2)


config.json:   0%|          | 0.00/950 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

...

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Model loaded and ready.


`torch.nn.functional.scaled_dot_product_attention` does not support `output_attentions=True`. Falling back to eager attention. This warning can be removed using the argument `attn_implementation="eager"` when loading the model.


vision tower layers 24


RuntimeError: a Tensor with 9 elements cannot be converted to Scalar